# MeteoScreening `LW_OUT_T1_47_1` (2020-2025) from database (influxdb)

***
**Site**: CH-LAE &nbsp;&nbsp;|&nbsp;&nbsp; **Variable**: `LW_OUT_T1_47_1` &nbsp;&nbsp;|&nbsp;&nbsp; **Sensor**: **Kipp & Zonen CNR1** to 14 Dec 2021, **CNR4** (SN 212965) after &nbsp;&nbsp;|&nbsp;&nbsp; **Period**: 2020-2025  
**Derived from**: diive notebook template `DatabaseInfluxStepwiseMeteoScreening.ipynb` (version `10`, 2 Sep 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Download raw outgoing longwave radiation from the InfluxDB database, remove what is
unambiguously wrong on the **high-resolution** data, resample to 30MIN, and upload the result
back to the database. Screening uses `StepwiseMeteoScreeningDb` from
[diive](https://github.com/holukas/diive) (`diive/preprocessing/qaqc/meteoscreening.py`);
download and upload use diive's in-house InfluxDB engine (`InfluxIO`, in
`diive/core/io/db/influx`).

**Flow:** download (`InfluxIO`) → screen on high-res data (`diive`) → resample to 30MIN → upload.

**Scope is deliberately narrow.** The raw record is over three million rows, so anything done
here is expensive. This notebook takes out the one episode that is not a measurement — the
CNR4 installation of 14 December 2021 — and hands over a clean 30MIN series. Everything else
(the level relationship to air temperature, the two calibration eras, the unverified
December 2021 window, gap handling, and any comparison against `LW_IN` or the subcanopy
sensor) is settled later in `30_PRODUCTS/`, **on the half-hourly data**, where it costs a
fraction as much.

**Outlier detection is stepwise:** run a test, inspect its preview plot, then commit it with
`mscr.addflag()`. Re-run with different parameters as often as you like before committing.
Run only the tests a variable actually needs. At the end all committed flags are aggregated
into one overall quality flag `QCF`.

> **What this variable is.** Outgoing (upwelling) longwave irradiance at 47 m: the thermal
> emission of the forest canopy below the radiometer, plus the reflected part of the
> downwelling flux. It is bounded well away from zero — a canopy at −10 °C still emits about
> 270 W m⁻² — and it tracks **surface temperature**, not solar geometry. Both facts drive
> every screening decision below, and both are what separate this variable from the shortwave
> channel it sits next to in every instrument and every variable list.

> **The raw field is the logger's corrected channel.** Its `raw_varname` tag is
> `LW_OUT_COR_T1_47_1_Avg`, so the temperature correction that turns a pyrgeometer's net
> signal into an irradiance has already been applied in the logger program. The values
> arriving here are irradiances, not raw thermopile voltages. This bears on the open question
> about `LW_OUT_COR_T1_47_1` in `ch-lae_processed` recorded in `PLAN.md`; establishing what
> that stored series is remains a `30_PRODUCTS/` question.

> **Database access** needs the `influxdb-client` package, which this project pulls in via the
> **`diive[db]` extra** (declared in `pyproject.toml`). diive *also* ships a `db` dependency
> group, but dependency groups are local to the project that declares them —
> `uv sync --group db` only works inside the diive repo, not from here.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`** (the stamp marks the
*end* of the averaging interval). Getting this right is the one thing that must not go wrong:
the value written back to the database depends on it, and so does the removal window in
`REMOVE_DATES`.

The single knob is `TIMEZONE_OFFSET_TO_UTC_HOURS` (set in *User settings*). It is applied
**identically** on download and on upload:

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| During screening | local | `TIMESTAMP_MIDDLE` (converted internally) | `StepwiseMeteoScreeningDb` |
| After `mscr.resample()` | local | back to `TIMESTAMP_END` | diive |
| After `dbc.upload_singlevar(..., timezone_offset_to_utc_hours=N)` | UTC | `TIMESTAMP_END` | InfluxIO |

**One place where this bites, and this notebook is exposed to it.** `REMOVE_DATES` is matched
against `TIMESTAMP_MID`, not `TIMESTAMP_END`: `StepwiseMeteoScreeningDb` hands the series to
`ManualRemoval` *after* the internal conversion, so on this 1MIN record the value whose
`TIMESTAMP_END` is `10:40:00` carries the label `10:39:30` inside the test — half a record
earlier. The window below is written on that shifted axis, and the audit re-derives its edges
from the data on every run rather than trusting the arithmetic.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site**
- `SITE`, `SITE_LAT`, `SITE_LON`: site ID and coordinates. The coordinates set the day/night
  split used during screening — not used here, see *Outlier detection*.

**Variable to screen**
- `FIELD`: the InfluxDB `_field`, assembled from the position tags. `FIELDS` is the list form
  `StepwiseMeteoScreeningDb` expects.
- `MEASUREMENT`: exactly **one** measurement grouping the variables — `LW` for longwave
  radiation, which at this site holds six fields across three locations.

**Time range to screen**
- `START`: first timestamp to screen — **is** included.
- `STOP`: upper bound — **is not** included.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the critical timestamp knob — see *Timestamp convention*.
  Must match how the raw data was logged (`1` for CET winter time) and must be the same value
  everywhere.
- `DATA_VERSION`: the source data version in the database (`raw`).
- `DIRCONF`: local folder holding the database connection config.

**Resampling**
- `RESAMPLING_FREQ` / `RESAMPLING_AGG`: an irradiance is a **rate**, so `'mean'` — never
  `'sum'`.

**Physical range**
- `LW_MIN`, `LW_MAX`: the absolute-limits test, justified from what a forest canopy can emit.

**Parameter help**
- `SHOW_PARAM_HELP`: `True` prints the full docstring of each screening method right before it
  runs.

In [ ]:
# --- Site ---
SITE = 'ch-lae'
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389  # CH-LAE

# --- Variable to screen ---
# Measurement LW holds six fields at three locations: the tower top (T1_47), the subcanopy
# station (BC_M1_2) and T3_1.5. This notebook screens the tower-top outgoing channel only.
VAR = 'LW_OUT'
HPOS = 'T1'  # horizontal position (tower)
VPOS = '47'  # measurement height, m
REPL = '1'
FIELD = f'{VAR}_{HPOS}_{VPOS}_{REPL}'
FIELDS = [FIELD]  # StepwiseMeteoScreeningDb expects a list
MEASUREMENT = 'LW'

# --- Time range to screen ---
# The raw record for this field begins 2020-01-02 00:52 and there is nothing before it: probes
# at 2005, 2008, 2011, 2014, 2016, 2018 and 2019 all return no data. So 2020 is the true start
# of the series, not a window chosen here, and LW_OUT (with ALB and a four-component NETRAD)
# can only ever be a 2020-2025 product. Asking for 2020-01-01 simply starts at the first record.
# STOP reaches one second into 2026 so that the last bin of 31 Dec 2025, whose TIMESTAMP_END is
# 2026-01-01 00:00, is included - it belongs to 2025.
START = '2020-01-01 00:00:01'  # included
STOP = '2026-01-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time). Must match how the raw data was logged.
DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Resampling ---
RESAMPLING_FREQ = '30min'  # screened high-res data is resampled to this frequency
RESAMPLING_AGG = 'mean'  # (!) an irradiance is a rate, never summed

# --- Physical range of outgoing longwave at this site, in W m-2 ---
# Derived from what the canopy can emit, not from the measured extremes. By Stefan-Boltzmann a
# grey surface near unit emissivity radiates 150 W m-2 at about -46 degC and 600 W m-2 at about
# +47 degC; neither is reachable by a mixed-forest canopy in the Swiss lowlands. Measured over
# 2020-2025, outside the December 2021 installation episode, this channel spans 259.5 to
# 508.2 W m-2, so the limits sit ~110 W m-2 below and ~90 W m-2 above anything it has produced.
# The test is a backstop against a disconnected or shorted thermopile, which rails rather than
# returning a plausible number - and it independently catches most of the December 2021 episode.
LW_MIN, LW_MAX = 150, 600

# --- Parameter help ---
SHOW_PARAM_HELP = False

## 🤖 Auto settings

### Buckets (do not adjust)

In [ ]:
BUCKET_RAW = f'{SITE}_raw'  # source bucket, e.g. 'ch-lae_raw'
BUCKET_PROCESSED = f'{SITE}_processed'  # destination bucket, e.g. 'ch-lae_processed'
print(f'Screening variable:             {FIELD}')
print(f'Source bucket (raw data):       {BUCKET_RAW}')
print(f'Destination bucket (processed): {BUCKET_PROCESSED}')

### Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import pandas as pd

import diive as dv
from diive.core.io.db.influx import InfluxIO  # needs influxdb-client, via the diive[db] extra

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional - list all fields available in the measurement (does not check the selected time
range). Worth a look here because `LW` groups **three different locations**: the tower top
(`_T1_47_1`), the subcanopy station (`_BC_M1_2_1`) and `_T3_1.5_1`. Those are separate
instruments looking at separate scenes and must never be spliced onto this one:

In [ ]:
display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Download
Returns three objects:
- `data_simple`: high-res time series, one column per variable (nice to look at).
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series **and its
  database tags** - this is what the screening consumes.
- `assigned_measurements`: the auto-detected measurement per variable (a sanity check).

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

### Inspect downloaded data

In [ ]:
data_simple

In [ ]:
assigned_measurements

Drop any requested variable that has no data in this period:

In [ ]:
vars_not_available = [v for v in FIELDS if v not in data_detailed.keys()]
for rem in vars_not_available:
    FIELDS.remove(rem)
    print(f'Removed {rem} from FIELDS (no data in this period).')
print(f'Data available for: {list(data_detailed.keys())}')
assert FIELD in data_detailed, f'(!) {FIELD} returned no data - nothing to screen'

### Inspect the database tags
The tags travel through screening onto the upload, so a change in any of them is a change in
what is being written back. This channel carries a **single** value for every tag over
2020-2025 — one ingest configuration, one gain, one offset, one unit — which is what makes the
whole period a single notebook. The `gain`/`offset` pair is checked against the magnitude of
the data rather than trusted: a `gain` tag records what was applied to that era's raw source,
and on its own it looks exactly like a factor bug.

In [ ]:
_tagcols = [c for c in data_detailed[FIELD].columns if c != FIELD]
for _c in _tagcols:
    _u = data_detailed[FIELD][_c].astype(str).unique()
    print(f'  {_c:20s} {list(_u)}')

# A single tag set is an assumption this notebook makes everywhere below - assert it.
_multi = {c: list(data_detailed[FIELD][c].astype(str).unique())
          for c in _tagcols if data_detailed[FIELD][c].astype(str).nunique() > 1}
assert not _multi, (f'(!) more than one value for tag(s) {_multi} - this period is NOT a single '
                    f'configuration and must be split into separate notebooks')
print('\n-> PASSED, one tag set over the whole period')

# Unit and scale: the stored values must be irradiances in W m-2, not thermopile voltages.
_u = data_detailed[FIELD]['units'].astype(str).iloc[0]
_med = data_detailed[FIELD][FIELD].median()
print(f"units tag: {_u!r}   median value: {_med:.2f}")
assert _u == 'W m-2', f'(!) unexpected units tag {_u!r}'
assert 200 < _med < 500, (f'(!) median {_med:.2f} is not an outgoing-longwave irradiance - '
                          f'check the gain/offset tags against the magnitude before screening')
print('-> PASSED, the magnitude matches the units tag')

### Verify download timestamps
Confirm the timestamps look right: **local time** (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`), marking
the **end** of each averaging interval. The database itself stores UTC - `InfluxIO` applied the
offset on download. Eyeball the first/last stamps against the `START`/`STOP` you requested.

In [ ]:
for v in data_detailed.keys():
    idx = data_detailed[v].index
    print(f'{v}: index name={idx.name!r}, tz={idx.tz}, freq={idx.freqstr}')
    print(f'   first={idx[0]}   last={idx[-1]}')
print(f'\nApplied UTC offset: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h (timestamps above are local time)')

### Coverage and time resolution
The record is **1MIN throughout** — unlike the soil profile, it has no resolution break at the
March 2021 logger rebuild, so there is no per-era reporting to do and no upsampled era in which
a differencing test would degenerate. The cell below states that rather than assuming it, and
lists the gaps.

In [ ]:
_s = data_detailed[FIELD][FIELD].dropna()
_step = pd.Series(_s.index).diff().dt.total_seconds().value_counts()
print('Spacing between consecutive records (seconds, top 5):')
print(_step.head(5).to_string())
assert _step.idxmax() == 60, f'(!) the dominant spacing is {_step.idxmax()} s, not 60 s'
print(f'\n-> 1MIN is {_step.max() / _step.sum() * 100:.2f} % of all steps: a single resolution era')

_grid = pd.date_range(_s.index[0].ceil('min'), _s.index[-1].floor('min'), freq='1min')
_present = _s.reindex(_grid).notna()
print(f'\nRecords present: {_present.sum():,} of {len(_grid):,} 1MIN slots '
      f'({_present.sum() / len(_grid) * 100:.2f} %)')

_gapid = (~_present != (~_present).shift()).cumsum()
_gaps = [(g.index[0], g.index[-1], len(g))
         for _, g in (~_present).groupby(_gapid) if g.iloc[0]]
_gaps = pd.DataFrame(_gaps, columns=['start', 'end', 'minutes'])
print(f'Gaps: {len(_gaps)} in total, {(_gaps.minutes > 1440).sum()} longer than a day')
print('\nThe five longest:')
print(_gaps.nlargest(5, 'minutes').to_string(index=False))

_cov = _present.groupby(_present.index.year).agg(['sum', 'size'])
_cov['pct'] = (_cov['sum'] / _cov['size'] * 100).round(2)
print('\nCoverage per year:')
print(_cov.to_string())

### Plot downloaded high-res data

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## ▶️ Start MeteoScreening with `diive`

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## 🔍 Outlier detection
Run a test → inspect its preview → commit with `mscr.addflag()`. Only the committed flag of the
**most recent** test is added. Skip any test a variable does not need.

**What is committed here**: a **manual window** over the CNR4 installation, a physical-range
backstop (**absolute limits**), and the **missing-values** flag. Why the rest are off:

- **Never split day and night.** This is the decision that most separates outgoing longwave
  from the shortwave channel of the same instrument. `SW_OUT` is zero at night by definition,
  so a solar-geometry split is exactly the right cut for it. `LW_OUT` is the canopy's own
  thermal emission: it is large at night, and its diurnal cycle follows **surface temperature**,
  which lags the sun by hours and stays elevated well past sunset. A solar split therefore cuts
  the series in the wrong place and only halves the sample behind every threshold. Every test
  below runs with `separate_day_night=False`. diive `v0.91.0` unified the name across
  every method; the older `separate_daytime_nighttime` raises an error naming its
  replacement.
- **Distribution-wide tests measure the season, not a fault.** A z-score over the whole record
  asks how far a value sits from the multi-year mean. Monthly means here run from about
  320 W m⁻² in January to about 415 W m⁻² in July; both ends are correct, and a z-score would
  spend its budget on summer afternoons.
- **Differencing-based tests would work here — and are still not committed.** The values are
  continuous (over 1.3 million distinct values in 3.15 million records, and the most frequent
  single value occurs 15 times), and there is one resolution era, so a Hampel filter has a real
  MAD to work with. They are left off because there is nothing for them to find once the manual
  window has run — see *Other tests* below, which measures that claim rather than asserting it.
- **Trim low does not apply.** It exists for radiation with a zero floor. Longwave has none;
  the smallest value this canopy has produced is about 260 W m⁻².

In [ ]:
mscr.start_outlier_detection()

Plot the current cleaned data at any point during detection:

In [ ]:
for key, val in mscr.outlier_detection.items():
    val.showplot_cleaned(interactive=False)

### Manual removal
Flag specific timestamps or time ranges for removal - known sensor failures, maintenance
windows, logger artefacts. Give `[start, stop]` pairs and/or single timestamps.

**This channel needs exactly one window: the CNR4 installation on 14 December 2021.** It is the
only episode in six years that is not a measurement, and the record itself dates it without
help from the maintenance log:

| time (local, `TIMESTAMP_END`) | what the record does |
|---|---|
| … 10:39 | ordinary and very smooth, drifting 313 → 316 W m⁻² over the morning |
| 10:40 | 294.5 W m⁻², a 21 W m⁻² step down within one minute |
| 10:41 - 13:22 | no records: 162 minutes, the third-longest gap in the whole series |
| 13:23 - 14:29 | values between 292 and **799 W m⁻²**, changing by up to 181 W m⁻² per minute |
| 14:30 - 14:33 | no records |
| 14:34 … | ordinary and very smooth again, 323 W m⁻² |

799 W m⁻² would require a surface at about 71 °C. What the instrument was looking at during
those minutes was the person installing it. The date is the one the maintenance record gives
for the CNR4 (see `docs/Instrumentation.md`), so the data and the fieldbook agree without
either having been fitted to the other.

**Why the absolute-limits test is not enough on its own.** It removes the 56 records above
600 W m⁻², but seven of the disturbed records sit *below* that line (down to 292 W m⁻²) while
still belonging to the same episode. The window removes the episode; the limits test is a
backstop that happens to agree with most of it.

**The window is written on `TIMESTAMP_MID`.** `ManualRemoval` sees labels shifted half a record
earlier than the `TIMESTAMP_END` stamps in the table above (see *Timestamp convention*), so the
edges below are `10:39:00` and `14:31:00`, which select the record stamped `10:40` and the one
stamped `14:29` while leaving the good records at `10:39` and `14:34` in place. The audit after
resampling re-derives both edges from the data on every run.

> ⚠️ **Do not copy this window into a sibling notebook.** A removal window belongs to a
> **sensor**. This one describes a person handling the tower-top radiometer; it says nothing
> about the subcanopy sensor or about `LW_IN`, whose own channel of the same instrument needs
> its own evidence.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.ManualRemoval)

In [ ]:
# The CNR4 installation, 14 December 2021. Edges are on TIMESTAMP_MID, i.e. half a record
# earlier than the TIMESTAMP_END stamps: 10:39:00 selects the record stamped 10:40:00, and
# 14:31:00 stops before the good record stamped 14:34:00. The audit below re-derives both.
# (!) Do not copy this window into a sibling notebook. A removal window belongs to a sensor.
REMOVE_DATES = [
    ['2021-12-14 10:39:00', '2021-12-14 14:31:00'],  # CNR4 installation, instrument handled
]

if REMOVE_DATES:
    mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)
else:
    print('No manual removal windows for this sensor - skip the addflag cell below.')

In [ ]:
if REMOVE_DATES:
    mscr.addflag()

### Absolute limits
Flags values outside a fixed physical range `[LW_MIN, LW_MAX]`. The limits come from what a
canopy can emit rather than from the measured extremes, so the test stays informative if the
record grows: by Stefan-Boltzmann, 150 W m⁻² is a surface near −46 °C and 600 W m⁻² one near
+47 °C.

Expect it to flag the high part of the December 2021 episode and nothing else. Because the
manual window above already covers that episode, most of what this test finds is a **second
opinion on records that are already gone** — which is the point: two independent criteria
identify the same minutes.

> ⚠️ **Do not clip instead.** `correction_setto_max_threshold` would turn a railed reading into
> a fabricated in-range value that nothing downstream could identify as fake. A removed value is
> honest; a clipped one is not.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.AbsoluteLimits)

In [ ]:
mscr.flag_outliers_abslim_test(minval=LW_MIN, maxval=LW_MAX, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Other tests (all off)
Left switched off because there is nothing for them to find, not because they would misbehave.
The cell below measures that claim on the raw series before any test has run, so it is checked
on every run rather than quoted from this paragraph.

The series is extremely smooth at 1MIN resolution: consecutive minutes typically differ by
about 0.2 W m⁻², and the largest one-minute change anywhere outside the December 2021 episode
is a **drop**, not a spike. Sudden falls of 10-18 W m⁻² are the physically expected sign — a
shower or a cloud shadow cooling a sun-warmed canopy within a minute — and they cluster on
convective summer afternoons. A spike test committed here would mostly rediscover that outgoing
longwave is smooth, and would put real weather at risk on exactly those afternoons.

If you do enable one: differencing-based tests compute their differences **after dropping
missing records**, so the records flanking each of the gaps listed above are compared across
the gap and look like spikes. Check what a test flags against the gap list before believing it.

In [ ]:
# Evidence for leaving the spike tests off - recomputed, not quoted.
# REMOVE_DATES is on TIMESTAMP_MID, the download on TIMESTAMP_END, so compare on MID.
_raw = data_detailed[FIELD][FIELD].dropna()
_raw.index = _raw.index - pd.Timedelta(seconds=30)
_consec = pd.Series(_raw.index).diff().dt.total_seconds().values == 60
_d = _raw.diff()[_consec]
_win = [(pd.Timestamp(a), pd.Timestamp(b)) for a, b in REMOVE_DATES]
_inwin = pd.Series(False, index=_d.index)
for _a, _b in _win:
    _inwin |= (_d.index >= _a) & (_d.index <= _b)

print('One-minute changes over the whole record:')
print(f'  median |change|                       {_d.abs().median():.3f} W m-2')
print(f'  99.9th percentile |change|            {_d.abs().quantile(.999):.2f} W m-2')
print(f'  largest |change| anywhere             {_d.abs().max():.2f} W m-2 '
      f'at {_d.abs().idxmax():%Y-%m-%d %H:%M}')

_out = _d[~_inwin]
_big = _out[_out.abs() > 10]
print(f'\nOutside the committed removal window(s):')
print(f'  largest |change|                      {_out.abs().max():.2f} W m-2 '
      f'at {_out.abs().idxmax():%Y-%m-%d %H:%M}')
print(f'  changes larger than 10 W m-2          {len(_big)}  '
      f'({int((_big < 0).sum())} negative, {int((_big > 0).sum())} positive)')
if len(_big):
    print('  their dates:', ', '.join(sorted({f'{d:%Y-%m-%d}' for d in _big.index})))
assert (_big > 0).sum() == 0, (
    '(!) a POSITIVE one-minute jump above 10 W m-2 survives outside the removal window. '
    'Outgoing longwave cannot rise that fast; investigate before trusting this record, and '
    'reconsider whether a spike test is now needed.')
print('\n-> no positive minute-to-minute jump outside the window: nothing for a spike test to find')

In [ ]:
# Optional, all off by default - read the note above before enabling any of these.
# mscr.flag_outliers_hampel_test(window_length=60 * 24, n_sigma=8, use_differencing=True,
#                                separate_day_night=False,
#                                repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_test(thres_zscore=4.5, separate_day_night=False,
#                                repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_rolling_test(thres_zscore=4.5, winsize=60 * 24 * 7,
#                                        repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_localsd_test(n_sd=7, winsize=60 * 24 * 7, constant_sd=False,
#                                 separate_day_night=False,
#                                 repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)
# mscr.addflag()

### Missing values
Not an outlier test - flags missing records so they are counted in the overall `QCF`.

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall quality flag QCF
Aggregate all committed test flags into one overall flag `QCF` (0 = good, 1 = marginal,
2 = bad) and filter the series. Required before corrections and resampling.

In [ ]:
mscr.finalize_outlier_detection()

#### Reports

In [ ]:
mscr.report_outlier_detection_qcf_evolution()

In [ ]:
mscr.report_outlier_detection_qcf_flags()

In [ ]:
mscr.report_outlier_detection_qcf_series()

#### Plots

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## 🔧 Corrections
Applied to the high-res, QCF-filtered data. **None of them apply to outgoing longwave**, and
every call below stays commented out.

> ⚠️ **The one that would do real damage: `correction_remove_nighttime_zero_offset()`.** It is
> the correction this variable is most likely to be given by mistake, because `LW_OUT` sits
> beside `SW_OUT` in the instrument, in the database measurement list and in the template's own
> section heading. That correction exists for **shortwave and PAR only** — `SW_IN`, `SW_OUT`,
> `PPFD_IN`, `PPFD_OUT` — where the true nighttime value is zero and any nonzero reading is a
> detector offset to be removed. Outgoing longwave at night is **not** zero: it is roughly
> 260-350 W m⁻² of canopy emission and is the larger part of the record. Applying the
> correction would set every nighttime record to zero and tilt the daytime ones by the
> interpolated slope, destroying more than half the series while leaving a plot that still
> looks like a diurnal cycle. It is never enabled in this notebook.

The remaining corrections are threshold clipping and setting ranges to a constant. Every one of
them writes a value the sensor did not measure, and nothing in this record needs one: the single
bad episode is removed, not repaired.

In [ ]:
mscr.showplot_cleaned()

Inspect the most frequent values first (read-only, safe to run). This channel is a converted
analogue signal, so expect a near-continuous distribution with no dominant value. A short list
of very frequent values here would be news: it would mean a stuck sensor, an unexpected
quantisation in the logger program, or a sentinel written on a failed read.

In [ ]:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} (top 20 of {mscr.series_hires_cleaned[ff].count():,} records, '
          f'{mscr.series_hires_cleaned[ff].nunique():,} distinct) ---')
    print(vc.head(20))

# A failed read that lands as a plausible number is the failure mode notna() cannot see. There
# is no such sentinel in this record, and this asserts it stays that way.
# Addressed by name rather than through the loop variables above: `ff` and `vc` would
# leave this check covering only the last field the moment FIELDS holds more than one.
_screened = mscr.series_hires_cleaned[FIELD]
_counts = _screened.value_counts()
_top_n, _top_v = _counts.iloc[0], _counts.index[0]
print(f'\nMost frequent single value: {_top_v:.4f} W m-2, {_top_n} times '
      f'({_top_n / _screened.count() * 100:.4f} % of records)')
assert _top_n < 0.001 * _screened.count(), (
    f'(!) the value {_top_v} occupies {_top_n} records - that is a stuck sensor or a sentinel, '
    f'not a measurement. Investigate before uploading.')
print('-> PASSED, the distribution is continuous: no stuck value and no sentinel')

In [ ]:
# All commented out on purpose - none of these apply to outgoing longwave. See the traps above.
# (!) NEVER the first one: nighttime LW_OUT is ~260-350 W m-2, not zero.
# mscr.correction_remove_nighttime_zero_offset()
# mscr.correction_remove_relativehumidity_offset()
# mscr.correction_setto_max_threshold(threshold=600)
# mscr.correction_setto_min_threshold(threshold=150)
# mscr.correction_setto_value(dates=[['2021-12-14', '2021-12-14']], value=0, verbose=1)
# mscr.correction_set_exact_value_to_missing(values=[0])
# mscr.showplot_cleaned(interactive=False)

## 📈 Analyses (optional)
`analysis_potential_radiation_correlation()` compares a measured series against potential
(clear-sky) radiation to reveal a timestamp shift. It is a **shortwave** diagnostic and is left
off: potential radiation is an astronomical quantity that goes to zero at night, whereas this
series does not, so the correlation would be dominated by the mismatch rather than by any
shift. If a timestamp shift has to be tested for this variable, the co-located `SW_IN` or
`PPFD_IN` channel is the right series to test it on, and that belongs in `30_PRODUCTS/`.

In [ ]:
# Shortwave diagnostic - not applicable to longwave. See the note above.
# _ = mscr.analysis_potential_radiation_correlation(utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
#                                                   mincorr=0.7, showplot=True)

## 🔁 Resampling

### Resample to 30MIN
Resample the screened high-res series to `RESAMPLING_FREQ`. The output timestamp is
`TIMESTAMP_END` again (see *Timestamp convention*), ready for upload. **This is the handover
point**: everything downstream of here works on the half-hourly series.

In [ ]:
# mincounts_perc stays at the template default of .25, i.e. 7.5 of the 30 records a full half
# hour holds at 1MIN. A 30MIN *mean* of a smooth quantity is well estimated from a quarter of
# the records - this series moves a few tenths of a W m-2 from one minute to the next. (A *sum*
# would be the opposite case and is why the precipitation notebook chose differently.)
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

### Check the resampled time resolution

In [ ]:
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: {freq}')

### 🧪 Audit the screening in both directions
`REMOVE_DATES` is not empty, so this notebook deletes data on a person's say-so and that is the
thing worth policing. The cell below asks the two questions the QCF reports cannot answer, and
prints the evidence for each on every run:

1. **Is everything that was removed genuinely bad?** The window must earn its place from the
   data, not from the table written above it. So its own justification is recomputed: the
   window must contain the record's extreme values, its minute-to-minute variability must stand
   far above the rest of the record, and the good records immediately outside its two edges must
   be ordinary — which is what shows the edges are in the right place. Separately, every record
   counted as out-of-range must really lie outside `[LW_MIN, LW_MAX]`, and none may survive.
2. **Is everything real still there?** Outside the window nothing may have been removed that no
   committed test asked for, no half hour may have been emptied by the removal of an isolated
   minute, and the level and spread of the series must be unchanged.

There is **no external reference** for this variable — MeteoSwiss carries no longwave channel
and NABEL has no pyrgeometer — so the second question cannot be answered by agreement with
another instrument, as it is for `SW_IN` or `SWC`. It is answered instead by comparing the
series against itself before and after screening, which is weaker, and the notebook says so
rather than implying a check it cannot perform.

In [ ]:
# Which index each object carries, because the audit compares them against each other:
#   data_detailed[FIELD]        TIMESTAMP_END  - the notebook's own download, never converted
#                                                (StepwiseMeteoScreeningDb copies the dict, so
#                                                 its internal conversion does not reach here)
#   mscr.series_hires_cleaned   TIMESTAMP_MID  - converted at init by TimestampSanitizer
#   mscr.resampled_detailed     TIMESTAMP_END  - resample_series_to_freq converts back
#   REMOVE_DATES                TIMESTAMP_MID  - matched against the converted series
_raw_hires = data_detailed[FIELD][FIELD].dropna()
_surv = mscr.series_hires_cleaned[FIELD].dropna()
_screened = mscr.resampled_detailed[FIELD][FIELD]
_windows = [(pd.Timestamp(a), pd.Timestamp(b)) for a, b in REMOVE_DATES]

# Put the raw series on the MID axis the removal test used, so the audit measures the records
# actually removed rather than their neighbours.
_mid = _raw_hires.copy()
_mid.index = _mid.index - pd.Timedelta(seconds=30)
_orig_mid = pd.DatetimeIndex(mscr.series_hires_orig[FIELD].dropna().index)
assert _mid.index.difference(_orig_mid).empty, (
    '(!) the shifted raw index does not line up with the series diive screened - the '
    'half-record offset assumed here is wrong, and every count below would be off by one record')

print('AUDIT 1 - is everything that was removed genuinely bad?')
print(f'  manual-removal windows committed:  {len(_windows)}')

_d_all = _mid.diff().abs()
for _a, _b in _windows:
    _in = _mid[_a:_b]
    _out = _mid.drop(_in.index)
    _before = _mid[:_a].iloc[-1] if len(_mid[:_a]) else float('nan')
    _after = _mid[_b:].iloc[0] if len(_mid[_b:]) else float('nan')
    print(f'    {_a:%Y-%m-%d %H:%M} -> {_b:%Y-%m-%d %H:%M}: {len(_in)} records '
          f'({len(_in) / len(_mid) * 100:.4f} % of the record)')
    print(f'       values                inside {_in.min():7.2f} .. {_in.max():7.2f}   '
          f'|  rest of record {_out.min():7.2f} .. {_out.max():7.2f}')
    print(f'       median |1-min change| inside {_d_all[_a:_b].median():7.2f}   '
          f'|  rest of record {_d_all.drop(_in.index).median():7.2f}')
    print(f'       nearest good records  before {_before:7.2f}   |  after {_after:7.2f}')
    _verdict = [_in.max() > _out.max(),                                     # holds the extremes
                _d_all[_a:_b].median() > 10 * _d_all.drop(_in.index).median(),  # far noisier
                LW_MIN < _before < LW_MAX, LW_MIN < _after < LW_MAX]        # edges land on good data
    if all(_verdict):
        print('       -> PASSED, the window holds the record extremes, is an order of magnitude '
              'noisier than the rest, and both edges sit next to ordinary values')
    else:
        print(f'       (!) this window does NOT stand out from the record (checks: {_verdict}) - '
              f're-justify it or drop it')

_oor = _raw_hires[(_raw_hires < LW_MIN) | (_raw_hires > LW_MAX)]
print(f'  out-of-range records in the raw series:  {len(_oor)}')
if len(_oor):
    print(f'     their value range:                    {_oor.min():.2f} .. {_oor.max():.2f} W m-2')
    print(f'     their dates:                          '
          f"{', '.join(sorted({f'{d:%Y-%m-%d}' for d in _oor.index}))}")
    assert ((_oor < LW_MIN) | (_oor > LW_MAX)).all(), \
        '(!) a record counted as out-of-range is actually inside the physical range'
assert _surv.between(LW_MIN, LW_MAX).all(), '(!) an out-of-range value survived screening'
print(f'  surviving hi-res values, all in [{LW_MIN}, {LW_MAX}]:  '
      f'{_surv.min():.2f} .. {_surv.max():.2f} W m-2  -> PASSED')

In [ ]:
print('AUDIT 2 - is everything real still there?')

# Nothing may be removed outside a committed window except what a committed test asked for.
# Both indexes below are on TIMESTAMP_MID already - series_hires_cleaned is not shifted again.
def _inside_any_window(idx):
    _m = pd.Series(False, index=idx)
    for _a, _b in _windows:
        _m |= (idx >= _a) & (idx <= _b)
    return _m.values

_cleaned_mid = pd.DatetimeIndex(mscr.series_hires_cleaned[FIELD].dropna().index)
_removed = _mid.index.difference(_cleaned_mid)
_in_any = _inside_any_window(_removed)
_oor_mid = pd.DatetimeIndex(_oor.index) - pd.Timedelta(seconds=30)
_unexplained = _removed[~_in_any].difference(_oor_mid)
print(f'  hi-res records removed in total:                {len(_removed)}')
print(f'     inside a manual-removal window:              {int(_in_any.sum())}')
print(f'     outside a window but out of physical range:  '
      f'{len(_removed) - int(_in_any.sum()) - len(_unexplained)}')
print(f'     removed with no committed test asking:       {len(_unexplained)}')
assert len(_unexplained) == 0, (
    f'(!) {len(_unexplained)} records left the series without a committed test asking for it, '
    f'first at {_unexplained[0] if len(_unexplained) else None}')

# Removing an isolated bad minute must not empty the half hour that held it. Out-of-range
# records lying inside a manual window are excluded - those half hours are meant to be gone.
_oor_outside = _oor_mid[~_inside_any_window(_oor_mid)]
_bins = (_oor_outside + pd.Timedelta(seconds=30)).ceil('30min').unique()  # back to END, then bin
print(f'  30MIN periods holding a removed out-of-range minute, outside any window: {len(_bins)}')
if len(_bins):
    _still = _screened.reindex(_bins)
    print(f'     of these, still holding a screened value:  {int(_still.notna().sum())}')
    assert _still.notna().all(), \
        '(!) removing an isolated out-of-range minute emptied the half hour that contained it'
else:
    print('     none - every out-of-range record lies inside the committed window')

# Level and spread must be unchanged outside the window: the screening removed one episode, so
# a shift in either would mean it also removed weather.
_ref = _mid[~_inside_any_window(_mid.index)]
_aft = mscr.series_hires_cleaned[FIELD].dropna().reindex(_ref.index).dropna()
print(f'  records outside the window, before -> after:    {len(_ref):,} -> {len(_aft):,}')
print(f'  mean   [W m-2], before -> after:                {_ref.mean():.4f} -> {_aft.mean():.4f}')
print(f'  std    [W m-2], before -> after:                {_ref.std():.4f} -> {_aft.std():.4f}')
assert abs(_ref.mean() - _aft.mean()) < 0.01 and abs(_ref.std() - _aft.std()) < 0.01, \
    '(!) screening moved the level or the spread outside the removal window - it removed weather'
print('  -> PASSED, outside the window the series is untouched')

# Resampling must not have invented or lost half hours.
print(f'\n  30MIN records after resampling:  {_screened.notna().sum():,} of {len(_screened):,} '
      f'({_screened.notna().sum() / len(_screened) * 100:.2f} %)')
assert _screened.dropna().between(LW_MIN, LW_MAX).all(), \
    '(!) a resampled 30MIN value lies outside the physical range'
print(f'  30MIN value range: {_screened.min():.2f} .. {_screened.max():.2f} W m-2  -> PASSED')

## ⬆️ Upload data to database

**Re-uploading overwrites the same variant - safe to re-run.** With
`delete_from_db_before_upload=True` (below), the upload first *deletes*, then writes. The delete
is scoped to the exact match `_measurement` + `varname` + `data_version`
(`meteoscreening_diive`) over the uploaded time range, so re-screening a period replaces only
its previous screened result. It never touches the raw data (different `data_version`, and a
different `_raw` bucket), other variables, or other data versions. The delete-first step (rather
than a plain overwrite) matters because InfluxDB keys a point by its full tag set: if a tag
changed between runs (e.g. `units`, `gain`, `offset`), a plain write would leave the old point
as a **duplicate** - the delete removes it regardless of tags.

> ℹ️ `ch-lae_processed` already holds a `LW_OUT_COR_T1_47_1` from earlier work. It is a
> **different field name**, so this upload neither reads nor replaces it.

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload
Download the just-uploaded data back and confirm the time resolution and timestamps. The offset
is the same `TIMEZONE_OFFSET_TO_UTC_HOURS`, so the timestamps below should again be local
`TIMESTAMP_END` - matching what you screened.

In [ ]:
# Fresh variable names so the screened originals (data_detailed etc.) are not overwritten:
check_simple, check_detailed, check_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
check_simple

In [ ]:
for v in check_detailed.keys():
    idx = check_detailed[v].index
    freq = dv.times.DetectFrequency(index=idx, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: freq={freq}, first={idx[0]}, last={idx[-1]}')

# What came back must be what went up.
_up = mscr.resampled_detailed[FIELD][FIELD].dropna()
_back = check_detailed[FIELD][FIELD].dropna()
_common = _up.index.intersection(_back.index)
print(f'\nuploaded {len(_up):,} records, read back {len(_back):,}, '
      f'{len(_common):,} on shared timestamps')
assert len(_common) > 0.99 * len(_up), '(!) the read-back is missing more than 1 % of the upload'
_dev = (_up.reindex(_common) - _back.reindex(_common)).abs().max()
print(f'largest deviation on shared timestamps: {_dev:.6f} W m-2')
assert _dev < 0.001, '(!) the values read back differ from those uploaded'
print('-> PASSED, the round trip preserved the values')

## ✅ End of notebook

In [ ]:
_end = datetime.now()
print(f"Finished: {_end.strftime('%Y-%m-%d %H:%M:%S')}  "
      f"(runtime {str(_end - NOTEBOOK_START).split('.')[0]})")

***
### 📝 Notes for this variable

- **Mean, not sum.** `RESAMPLING_AGG = 'mean'` - an irradiance is a rate, not an accumulation.
- **Units are W m⁻²**, filegroup `10_meteo`, `gain` 1.0 and `offset` 0.0 - one tag set over the
  whole period, asserted after download.
- **The raw field is the logger's corrected channel.** `raw_varname` is
  `LW_OUT_COR_T1_47_1_Avg`: the pyrgeometer's temperature correction is applied in the logger
  program, so these values are irradiances, not raw thermopile voltages. What the separately
  stored `LW_OUT_COR_T1_47_1` in `ch-lae_processed` is, and how it relates to this field, is an
  open question in `PLAN.md` and belongs in `30_PRODUCTS/`.
- **The record begins 2020-01-02 00:52**, and probes at 2005, 2008, 2011, 2014, 2016, 2018 and
  2019 return nothing. The tower's outgoing longwave channel was therefore not archived before
  2020, which settles the period of the future `LW_OUT` product - and, with the matching
  question for `SW_OUT`, of `ALB` and any four-component `NETRAD`: **2020-2025, not 2005-2025**.
  This answers one of the open questions in `PLAN.md` section 6.
- **Coverage** 3,151,108 records, 99.88 % of the 1MIN grid over 2020-2025. 39 gaps, 33 of them
  an hour or shorter; only one exceeds a day (2024-06-29 17:19 to 2024-07-01 13:24, 1.8 days),
  which is why 2024 is the only year below 99.9 %.
- **A single 1MIN resolution era**, with no break at the March 2021 logger rebuild. The per-era
  reporting the soil-profile notebooks insist on is a no-op here, and no era arrives upsampled,
  so a differencing test would have a real scale to work with.
- **Measured range 259.5 … 508.2 W m⁻²** outside the December 2021 episode, median
  361.3 W m⁻². The extremes are physically ordinary: 259.5 W m⁻² is a canopy near −13 °C on a
  February morning, 508.2 W m⁻² one near +34 °C on the afternoon of 4 August 2022, during that
  summer's heatwave.
- **Values are continuous**, over 1.3 million distinct in 3.15 million records, and the most
  frequent single value occurs 15 times. There is no sentinel for a failed read in this record
  and no quantisation, which is asserted rather than assumed.
- **One removal window: the CNR4 installation on 14 December 2021**, 10:40 to 14:29 local time,
  63 records reaching 799 W m⁻². The instrument was being handled. The date the data give is the
  date the maintenance record gives, independently.
- **The series spans two radiometers** - CNR1 until 14 December 2021, CNR4 (SN 212965,
  `LW_OUT` sensitivity 11.33 µV W⁻¹ m²) after - and the logger program did not receive the CNR4
  constants until **7 January 2022**. Those 24 days are flagged unverified in
  `docs/Instrumentation.md`. Nothing is done about it here: this notebook removes artefacts, and
  a calibration era is not an artefact. Daily means across both dates wander with the weather
  and show no obvious step, but that is an observation, not a test - deciding it needs air
  temperature as a covariate and belongs in `30_PRODUCTS/`, on the 30MIN data.
- **No external reference exists for this variable.** MeteoSwiss Lägern carries no longwave
  channel and NABEL has no pyrgeometer, so every check in this notebook is internal. That is
  weaker than the cross-checks available to `SW_IN` or `SWC`, and the audit says so.
- **Deferred to `30_PRODUCTS/`, on the 30MIN data:** the two calibration eras and whether they
  need a `SOURCE` flag, the unverified December 2021 window, the relation to air temperature and
  to `LW_IN`, the comparison against the subcanopy `LW_OUT_BC_M1_2_1`, and any gap handling.
  This notebook exports gaps where the station was down and does not fill them.

### ♻️ Sibling notebooks
`LW_IN/` screens the **other channel of the same instrument** and is split into
`LW_IN_T1_47_1_2022-2023` and `LW_IN_T1_47_1_2025`, both from an older template version. Two
things do **not** carry across from it. Its 7 June 2016 calibration error predates this record
entirely, so no part of it applies here. And its removal windows describe the downward-facing
channel; this window describes a person in front of the upward-looking one on the day it was
mounted.

The standing rule for all of them: **`REMOVE_DATES` belongs to a sensor, not to a variable.**
These notebooks are copies of each other with a few settings changed, and carrying a window
across a copy deletes good data under a justification written about a different instrument.